# Черновая работа

In [1]:
import pandas as pd

df = pd.read_excel('/Users/gregorlov/Downloads/bank_data.xlsx')

print(df.shape)
print(df.dtypes)
df.head(15)


(1460, 9)
date                   datetime64[us]
regn                              str
Assets                         object
Loans_Total_Net                object
Loans_LLP                      object
Equity                         object
Net_Income_Current             object
Client_Deposits                object
Net_Interest_Income            object
dtype: object


,date,regn,Assets,Loans_Total_Net,Loans_LLP,Equity,Net_Income_Current,Client_Deposits,Net_Interest_Income
0,2026-01-01,bank_1,602749.6,51793.8,-15287.6,306245.5,34328.9,271026.2,13631.7
1,2025-12-01,bank_1,651921,57052.6,-18099.5,315195,43079,313635,NaN
2,2025-11-01,bank_1,634206.6,59859.9,-20773.4,306073,32517.3,304062,NaN
3,2025-10-01,bank_1,639777.8,63138.4,-22815.3,299638.7,27023,312620.8,NaN
4,2025-09-01,bank_1,658239.2,65308.3,-22476.6,311847.4,41208.7,315765.1,NaN
5,2025-08-01,bank_1,656141.9,67345.2,-23098.6,307400.3,38703.9,318345.4,NaN
6,2025-07-01,bank_1,643574.3,69370.9,-22221.8,301844.6,34152.1,311073.5,16306
7,2025-06-01,bank_1,668944.6,72843.7,-22315.5,295615.5,29053.5,339030,NaN
8,2025-05-01,bank_1,"729 873,3 млн руб.",77498.8,-21921.9,289603.3,25766.7,365471.1,NaN
9,2025-04-01,bank_1,731308.7,85697.3,-21434.1,285859.6,22108.7,373633.9,17481


In [2]:
print('банков:', df['regn'].nunique(), '|', df['regn'].unique())
print('период:', df['date'].min(), '->', df['date'].max())
print('строк на банк:')
print(df.groupby('regn').size().describe())

банков: 20 | <StringArray>
[ 'bank_1',  'bank_2',  'bank_3',  'bank_4',  'bank_5',  'bank_6',  'bank_7',
  'bank_8',  'bank_9', 'bank_10', 'bank_11', 'bank_12', 'bank_13', 'bank_14',
 'bank_15', 'bank_16', 'bank_17', 'bank_18', 'bank_19', 'bank_20']
Length: 20, dtype: str
период: 2020-01-01 00:00:00 -> 2026-01-01 00:00:00
строк на банк:
count    20.0
mean     73.0
std       0.0
min      73.0
25%      73.0
50%      73.0
75%      73.0
max      73.0
dtype: float64


In [3]:
df.isna().sum() 

date                     0
regn                     0
Assets                   6
Loans_Total_Net          2
Loans_LLP                2
Equity                   4
Net_Income_Current       1
Client_Deposits          2
Net_Interest_Income    964
dtype: int64

In [4]:
num_cols = ['Assets','Loans_Total_Net','Loans_LLP','Equity',
            'Net_Income_Current','Client_Deposits','Net_Interest_Income']
for c in num_cols:
    if df[c].dtype == object:
        bad = df[pd.to_numeric(df[c], errors='coerce').isna() & df[c].notna()]
        print(f'\n--- {c}: {len(bad)} «грязных» значений ---')
        print(bad[['date','regn',c]].head(10))



--- Assets: 5 «грязных» значений ---
           date     regn                   Assets
8    2025-05-01   bank_1       729 873,3 млн руб.
403  2022-11-01   bank_6        97 513,0 млн руб.
526  2024-10-01   bank_8       186 818,5 млн руб.
896  2024-05-01  bank_13  272 267 300,0 тыс. руб.
1459 2020-01-01  bank_20           667,0 млн руб.

--- Loans_Total_Net: 6 «грязных» значений ---
          date     regn           Loans_Total_Net
30  2023-07-01   bank_1  -321 279 200,0 тыс. руб.
49  2021-12-01   bank_1        583 657,3 млн руб.
145 2020-01-01   bank_2      8 329 747,9 млн руб.
260 2022-08-01   bank_4    36 327 500,0 тыс. руб.
553 2022-07-01   bank_8    78 821 500,0 тыс. руб.
963 2024-11-01  bank_14        790 843,9 млн руб.

--- Loans_LLP: 6 «грязных» значений ---
           date     regn                Loans_LLP
76   2025-10-01   bank_2    -1 037 343,1 млн руб.
190  2022-05-01   bank_3  -23 657 100,0 тыс. руб.
330  2022-11-01   bank_5          -244,9 млн руб.
461  2024-02-01   bank_7

In [5]:
for c in ['Equity','Client_Deposits','Assets']:
    s = pd.to_numeric(df[c], errors='coerce')
    print(c, 'min=', s.min(), 'neg=', (s<0).sum(), 'zero=', (s==0).sum())


Equity min= -367346.3 neg= 172 zero= 0
Client_Deposits min= -12746038.2 neg= 5 zero= 0
Assets min= -992753.5 neg= 3 zero= 0


In [6]:
df['month'] = pd.to_datetime(df['date']).dt.month
df.groupby('month')['Net_Interest_Income'].apply(lambda s: s.notna().mean())


month
1     0.992857
2     0.000000
3     0.000000
4     1.000000
5     0.000000
6     0.000000
7     0.991667
8     0.000000
9     0.000000
10    0.983333
11    0.000000
12    0.000000
Name: Net_Interest_Income, dtype: float64

In [7]:
b = df[df['regn']=='bank_1'].sort_values('date')
b[['date','Net_Income_Current']].head(30)


,date,Net_Income_Current
72,2020-01-01,7872.6
71,2020-02-01,2753.7
70,2020-03-01,4097.4
69,2020-04-01,4529.6
68,2020-05-01,4702.2
67,2020-06-01,5813.9
66,2020-07-01,2431
65,2020-08-01,2296.2
64,2020-09-01,5955.4
63,2020-10-01,5953.4


In [8]:
b = df[df['regn']=='bank_1'].sort_values('date').copy()
b['NIC'] = pd.to_numeric(b['Net_Income_Current'], errors='coerce')

b['year'] = b['date'].dt.year
print('--- сумма NIC по годам (если monthly — это годовая прибыль) ---')
print(b.groupby('year')['NIC'].sum())

print('\n--- монотонность внутри года (если YTD — diff внутри года >= 0) ---')
b['ytd_diff'] = b.groupby('year')['NIC'].diff()
print('отрицательных diff внутри года:', (b['ytd_diff'] < 0).sum(), 'из', b['ytd_diff'].notna().sum())


--- сумма NIC по годам (если monthly — это годовая прибыль) ---
year
2020     67326.1
2021    105415.0
2022    248826.4
2023    300261.3
2024    337112.8
2025    354861.3
2026     34328.9
Name: NIC, dtype: float64

--- монотонность внутри года (если YTD — diff внутри года >= 0) ---
отрицательных diff внутри года: 14 из 66


In [9]:
import re
import numpy as np
import pandas as pd

NBSP = '\u00a0'
NARROW_NBSP = '\u202f'


def parse_value(x):
    '''Приводим значение к float в единицах млн руб.
    Будут корректно парситься  числа, 'NNN,N', 'N NNN,N млн руб.', 'N NNN NNN,N тыс. руб.
    Для пустых/нераспознанных значений будет возвращаться np.nan
    '''
    if pd.isna(x):
        return np.nan 
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace(NBSP, ' ').replace(NARROW_NBSP, ' ')
    unit_mult = 1.0 
    low = s.lower()
    if 'тыс' in low:
        unit_mult = 1e-3
        s = re.sub(r'тыс\.?\s*руб\.?', '', s, flags=re.IGNORECASE)
    elif 'млн' in low:
        unit_mult = 1.0
        s = re.sub(r'млн\.?\s*руб\.?', '', s, flags=re.IGNORECASE)
    elif 'млрд' in low:
        unit_mult = 1e3
        s = re.sub(r'млрд\.?\s*руб\.?', '', s, flags=re.IGNORECASE)
    s = s.replace(' ', '').replace(',', '.').strip()
    if s in ('', '-', 'nan', 'None'):
        return np.nan
    return float(s) * unit_mult 



In [10]:
clean = df.copy()

for c in num_cols:
    before_nan = clean[c].isna().sum()
    clean[c] = clean[c].apply(parse_value)
    after_nan = clean[c].isna().sum()
    print(f'{c} было NaN={before_nan}, стало NaN={after_nan}, восстановлено из текста={before_nan - after_nan if after_nan<=before_nan else "—"}')

Assets было NaN=6, стало NaN=6, восстановлено из текста=0
Loans_Total_Net было NaN=2, стало NaN=2, восстановлено из текста=0
Loans_LLP было NaN=2, стало NaN=2, восстановлено из текста=0
Equity было NaN=4, стало NaN=4, восстановлено из текста=0
Net_Income_Current было NaN=1, стало NaN=1, восстановлено из текста=0
Client_Deposits было NaN=2, стало NaN=2, восстановлено из текста=0
Net_Interest_Income было NaN=964, стало NaN=964, восстановлено из текста=0


In [11]:
if 'regn' not in clean.columns:
    clean['regn'] = df['regn'].values
clean = clean.sort_values(['regn', 'date']).reset_index(drop=True)


In [12]:
test_cases = [
    ('729 873,3 млн руб.',         729873.3),
    ('272 267 300,0 тыс. руб.',    272267.3),     
    ('-321 279 200,0 тыс. руб.',   -321279.2),     
    ('11 414 945 300,0 тыс. руб.', 11414945.3),   
    ('667,0 млн руб.',             667.0),
    ('-244,9 млн руб.',            -244.9),
    (602749.6,                     602749.6),      
    (None,                         None),       
]
for raw, expected in test_cases:
    got = parse_value(raw)
    ok = (expected is None and pd.isna(got)) or (expected is not None and abs(got - expected) < 1e-6)
    print(f'{"OK" if ok else "FAIL"}  {raw!r:40} -> {got!r} (ожидали {expected!r})')


OK  '729 873,3 млн руб.'                     -> 729873.3 (ожидали 729873.3)
OK  '272 267 300,0 тыс. руб.'                -> 272267.3 (ожидали 272267.3)
OK  '-321 279 200,0 тыс. руб.'               -> -321279.2 (ожидали -321279.2)
OK  '11 414 945 300,0 тыс. руб.'             -> 11414945.3 (ожидали 11414945.3)
OK  '667,0 млн руб.'                         -> 667.0 (ожидали 667.0)
OK  '-244,9 млн руб.'                        -> -244.9 (ожидали -244.9)
OK  602749.6                                 -> 602749.6 (ожидали 602749.6)
OK  None                                     -> nan (ожидали None)


In [13]:
# Пример: bank_13 2024-05-01 был '272 267 300,0 тыс. руб.'
b = clean[clean['regn']=='bank_13'].sort_values('date')
print(b.loc[b['date'].between('2024-03-01','2024-07-01'), ['date','Assets']])
# Ожидаем: Assets в мае ≈ того же порядка, что в апр/июн


          date    Assets
342 2024-03-01  268034.3
343 2024-04-01  264620.7
344 2024-05-01  272267.3
345 2024-06-01  280539.0
346 2024-07-01  274314.2


In [14]:
# Приведём все числовые колонки к float (должны уже быть, но на всякий случай)
for c in num_cols:
    clean[c] = pd.to_numeric(clean[c], errors='coerce')

# Флаги ошибок данных и методологические
clean['flag_equity_negative']   = clean['Equity'] < 0                          # блокирует ROE (по ТЗ)
clean['flag_loans_negative']    = clean['Loans_Total_Net'] < 0                 # ошибка данных → блокирует LDR, Loan Yield
clean['flag_deposits_negative'] = clean['Client_Deposits'] < 0                 # ошибка данных → блокирует LDR
clean['flag_assets_negative']   = clean['Assets'] < 0                          # информационный (в коэффициентах не участвует)
# Резерв по модулю больше самих кредитов — подозрение на ошибку величины/знака LLP
clean['flag_llp_exceeds_net']   = clean['Loans_LLP'].abs() > clean['Loans_Total_Net'].abs()
# LLP по идее отрицательный (резерв). Если положительный — подозрительно
clean['flag_llp_positive_sign'] = clean['Loans_LLP'] > 0

flag_cols = [c for c in clean.columns if c.startswith('flag_')]
print('Сводка по флагам (всего 1460 строк):')
print(clean[flag_cols].sum().to_frame('кол-во строк'))

print('\nРазбивка по банкам:')
flags_by_bank = clean.groupby('regn')[flag_cols].sum()
print(flags_by_bank[flags_by_bank.sum(axis=1) > 0])


Сводка по флагам (всего 1460 строк):
                        кол-во строк
flag_equity_negative             172
flag_loans_negative                7
flag_deposits_negative             5
flag_assets_negative               3
flag_llp_exceeds_net              17
flag_llp_positive_sign             0

Разбивка по банкам:
         flag_equity_negative  flag_loans_negative  flag_deposits_negative  \
regn                                                                         
bank_1                      0                    1                       0   
bank_10                    27                    0                       0   
bank_11                    73                    1                       0   
bank_13                     0                    0                       1   
bank_14                     0                    0                       0   
bank_16                     0                    1                       0   
bank_17                     0                    1         

In [15]:
# Для удобства: сортировка и подготовка
clean = clean.sort_values(['regn','date']).reset_index(drop=True)
clean['month'] = clean['date'].dt.month

# Gross loans = Net - LLP (LLP отрицательный, значит Gross > Net)
clean['Loans_Gross'] = clean['Loans_Total_Net'] - clean['Loans_LLP']

# --- ROE 12m ---
# profit_12m = rolling_sum(Net_Income_Current, 12)  — НЕ включая текущий месяц? По ТЗ "за последние 12 месяцев"
# Берём 12 последних включая текущий: window=12, min_periods=12
# equity_avg = rolling_mean(Equity, 13 точек) — от t-12 до t включительно
g = clean.groupby('regn', group_keys=False)

clean['profit_12m'] = g['Net_Income_Current'].apply(
    lambda s: s.rolling(window=12, min_periods=12).sum()
)
clean['equity_avg_13'] = g['Equity'].apply(
    lambda s: s.rolling(window=13, min_periods=13).mean()
)

# Если Equity(t) < 0 по ТЗ, то не считаем. Плюс если в окне Equity любого периода < 0 —
# капитал знакопеременный, среднее теряет смысл; тоже не считаем.
clean['equity_any_neg_in_window'] = g['Equity'].apply(
    lambda s: (s < 0).rolling(window=13, min_periods=13).sum() > 0
)

mask_roe_ok = (
    clean['profit_12m'].notna() &
    clean['equity_avg_13'].notna() &
    ~clean['equity_any_neg_in_window'].fillna(True) &
    (clean['Equity'] >= 0)
)
clean['ROE_12m'] = np.where(mask_roe_ok, clean['profit_12m'] / clean['equity_avg_13'], np.nan)

# --- LDR (на каждую дату) ---
mask_ldr_ok = (
    (clean['Loans_Gross'] >= 0) &
    (clean['Client_Deposits'] > 0)
)
clean['LDR'] = np.where(mask_ldr_ok, clean['Loans_Gross'] / clean['Client_Deposits'], np.nan)

# --- Доходность кредитного портфеля 3m (только для квартальных дат) ---
# loans_avg_Q = mean за 4 точки квартала (t, t-1, t-2, t-3 мес)
clean['loans_avg_Q'] = g['Loans_Total_Net'].apply(
    lambda s: s.rolling(window=4, min_periods=4).mean()
)
# в окне не должно быть отрицательных Loans
clean['loans_any_neg_in_Q'] = g['Loans_Total_Net'].apply(
    lambda s: (s < 0).rolling(window=4, min_periods=4).sum() > 0
)

is_qdate = clean['month'].isin([1,4,7,10])
mask_yield_ok = (
    is_qdate &
    clean['Net_Interest_Income'].notna() &
    clean['loans_avg_Q'].notna() &
    ~clean['loans_any_neg_in_Q'].fillna(True) &
    (clean['loans_avg_Q'] > 0)
)
clean['LoanYield_3m'] = np.where(
    mask_yield_ok,
    4 * clean['Net_Interest_Income'] / clean['loans_avg_Q'],
    np.nan
)

# Сводка покрытия
print('Строк с посчитанным ROE:', clean['ROE_12m'].notna().sum(), 'из 1460')
print('Строк с посчитанным LDR:', clean['LDR'].notna().sum(), 'из 1460')
print('Строк с посчитанным LoanYield:', clean['LoanYield_3m'].notna().sum(),
      'из', is_qdate.sum(), 'квартальных дат')


Строк с посчитанным ROE: 1032 из 1460
Строк с посчитанным LDR: 1442 из 1460
Строк с посчитанным LoanYield: 464 из 500 квартальных дат


In [16]:
# Sanity: распределение коэффициентов
print('=== ROE_12m ===')
print(clean['ROE_12m'].describe([.01,.05,.5,.95,.99]))
print('\n=== LDR ===')
print(clean['LDR'].describe([.01,.05,.5,.95,.99]))
print('\n=== LoanYield_3m ===')
print(clean['LoanYield_3m'].describe([.01,.05,.5,.95,.99]))

# Outliers — что за пределами «разумных» диапазонов
print('\n--- ROE вне [-1, 1]: ---', (clean['ROE_12m'].abs() > 1).sum())
print('--- LDR вне [0, 5]:  ---', ((clean['LDR'] < 0) | (clean['LDR'] > 5)).sum())
print('--- LoanYield вне [0, 0.5]: ---',
      ((clean['LoanYield_3m'] < 0) | (clean['LoanYield_3m'] > 0.5)).sum())


=== ROE_12m ===
count    1032.000000
mean        1.107932
std         1.886949
min        -9.650865
1%         -3.810906
5%         -0.432156
50%         0.874622
95%         3.754581
99%         9.016060
max        12.167754
Name: ROE_12m, dtype: float64

=== LDR ===
count    1442.000000
mean        0.795220
std         0.326051
min         0.045910
1%          0.126387
5%          0.287643
50%         0.816289
95%         1.237520
99%         1.704418
max         3.014135
Name: LDR, dtype: float64

=== LoanYield_3m ===
count    464.000000
mean       0.165280
std        0.501543
min       -0.372534
1%        -0.138987
5%        -0.034538
50%        0.070993
95%        0.563532
99%        1.549383
max        5.664554
Name: LoanYield_3m, dtype: float64

--- ROE вне [-1, 1]: --- 477
--- LDR вне [0, 5]:  --- 0
--- LoanYield вне [0, 0.5]: --- 57


In [17]:
latest = clean[clean['date'] == clean['date'].max()][
    ['regn','date','ROE_12m','LDR','LoanYield_3m',
     'Equity','Loans_Total_Net','Loans_Gross','Client_Deposits']
].sort_values('regn', key=lambda s: s.str.extract(r'(\d+)').astype(int)[0])

# Для читаемости — в процентах
latest_fmt = latest.copy()
latest_fmt['ROE_12m'] = (latest_fmt['ROE_12m']*100).round(2)
latest_fmt['LoanYield_3m'] = (latest_fmt['LoanYield_3m']*100).round(2)
latest_fmt['LDR'] = latest_fmt['LDR'].round(3)
latest_fmt.rename(columns={'ROE_12m':'ROE_12m,%', 'LoanYield_3m':'LoanYield_3m,%'})


,regn,date,"ROE_12m,%",LDR,"LoanYield_3m,%",Equity,Loans_Total_Net,Loans_Gross,Client_Deposits
72,bank_1,2026-01-01,119.85,0.248,94.07,306245.5,51793.8,67081.4,271026.2
875,bank_2,2026-01-01,109.21,0.781,2.90,1788345.6,17387321.9,18433133.4,23610335.1
1021,bank_3,2026-01-01,NaN,1.496,34.96,-305538.0,660281.8,760043.2,507905.5
1094,bank_4,2026-01-01,62.84,0.880,10.58,13697.6,56796.5,62962.4,71574.6
1167,bank_5,2026-01-01,301.93,0.617,21.68,43930.0,133936.9,135752.0,219852.2
1240,bank_6,2026-01-01,35.20,1.070,5.71,19774.2,110717.1,113425.7,106029.3
1313,bank_7,2026-01-01,116.41,0.848,13.69,104925.4,407387.7,439321.4,518311.6
1386,bank_8,2026-01-01,38.19,0.902,9.12,31785.6,57253.3,94925.8,105247.8
1459,bank_9,2026-01-01,118.19,0.851,6.18,372353.7,2532882.4,2627813.4,3086646.7
145,bank_10,2026-01-01,246.06,0.699,15.48,4129.5,31703.7,37076.5,53028.0


In [18]:
b1 = clean[clean['regn']=='bank_1'].sort_values('date').tail(13).copy()
print(b1[['date','Net_Income_Current','Equity']].to_string())
print('\nsum NIC за 12 мес (фев25..янв26):', b1['Net_Income_Current'].tail(12).sum())
print('mean Equity за 13 точек:', b1['Equity'].mean())
print('ROE = ', b1['Net_Income_Current'].tail(12).sum() / b1['Equity'].mean())
print('\nВ коде ROE_12m на 2026-01-01:', b1['ROE_12m'].iloc[-1])


         date  Net_Income_Current    Equity
60 2025-01-01             35664.9  262082.3
61 2025-02-01              9278.6  273087.4
62 2025-03-01             16304.9  280196.3
63 2025-04-01             22108.7  285859.6
64 2025-05-01             25766.7  289603.3
65 2025-06-01             29053.5  295615.5
66 2025-07-01             34152.1  301844.6
67 2025-08-01             38703.9  307400.3
68 2025-09-01             41208.7  311847.4
69 2025-10-01             27023.0  299638.7
70 2025-11-01             32517.3  306073.0
71 2025-12-01             43079.0  315195.0
72 2026-01-01             34328.9  306245.5

sum NIC за 12 мес (фев25..янв26): 353525.30000000005
mean Equity за 13 точек: 294976.06923076924
ROE =  1.198488070309954

В коде ROE_12m на 2026-01-01: 1.1984880703099539


In [19]:
clean['flag_roe_outlier']   = clean['ROE_12m'].abs() > 1
clean['flag_yield_outlier'] = (clean['LoanYield_3m'] < 0) | (clean['LoanYield_3m'] > 0.5)
clean['flag_ldr_outlier']   = (clean['LDR'] < 0.1) | (clean['LDR'] > 3)

print('Outliers:')
print(f"  ROE |>1|:        {clean['flag_roe_outlier'].sum():4} из {clean['ROE_12m'].notna().sum()}")
print(f"  LoanYield вне [0,50%]: {clean['flag_yield_outlier'].sum():4} из {clean['LoanYield_3m'].notna().sum()}")
print(f"  LDR вне [0.1, 3]:  {clean['flag_ldr_outlier'].sum():4} из {clean['LDR'].notna().sum()}")

# Топ-5 нереалистичных ROE
print('\nТоп-5 по |ROE|:')
print(clean.nlargest(5, 'ROE_12m', keep='all')[['date','regn','ROE_12m','profit_12m','equity_avg_13']].to_string())


Outliers:
  ROE |>1|:         477 из 1032
  LoanYield вне [0,50%]:   57 из 464
  LDR вне [0.1, 3]:     8 из 1442

Топ-5 по |ROE|:
          date     regn    ROE_12m  profit_12m  equity_avg_13
936 2025-01-01  bank_20  12.167754     80308.3    6600.092308
112 2023-04-01  bank_10  12.091640      8601.9     711.392308
935 2024-12-01  bank_20  11.960499     61401.8    5133.715385
934 2024-11-01  bank_20  11.716577     46733.1    3988.630769
933 2024-10-01  bank_20  11.357198     34290.7    3019.292308


In [20]:
b1 = clean[clean['regn']=='bank_1'].sort_values('date').copy()
b1['year'] = b1['date'].dt.year

# Под flow: годовая прибыль = сумма 12 месяцев
sum_by_year = b1.groupby('year')['Net_Income_Current'].sum()

# Под YTD: годовая прибыль года N = значение на 1 января N+1
jan_values = b1[b1['date'].dt.month == 1].set_index(b1.loc[b1['date'].dt.month == 1, 'year'])['Net_Income_Current']
jan_as_fy_prev = jan_values.shift(-1)  # если 1 янв 2021 = YTD FY2020 — сдвигаем на год назад

cmp = pd.DataFrame({
    'sum_year (flow)': sum_by_year,
    'Jan_next_year (YTD FY)': jan_as_fy_prev,
})
cmp['ratio flow/YTD'] = cmp['sum_year (flow)'] / cmp['Jan_next_year (YTD FY)']
print(cmp)


      sum_year (flow)  Jan_next_year (YTD FY)  ratio flow/YTD
year                                                         
2020          67326.1                  8989.8        7.489165
2021         105415.0                 13154.8        8.013425
2022         248826.4                 50581.5        4.919316
2023         300261.3                 50965.1        5.891508
2024         337112.8                 35664.9        9.452229
2025         354861.3                 34328.9       10.337101
2026          34328.9                     NaN             NaN


In [21]:
if 'regn' not in clean.columns:
    clean = clean.reset_index(drop=True)
    clean['regn'] = df.sort_values(['regn','date']).reset_index(drop=True)['regn'].values
print(clean.columns.tolist())
print(clean.groupby('regn').size().head(3))


['date', 'regn', 'Assets', 'Loans_Total_Net', 'Loans_LLP', 'Equity', 'Net_Income_Current', 'Client_Deposits', 'Net_Interest_Income', 'month', 'flag_equity_negative', 'flag_loans_negative', 'flag_deposits_negative', 'flag_assets_negative', 'flag_llp_exceeds_net', 'flag_llp_positive_sign', 'Loans_Gross', 'profit_12m', 'equity_avg_13', 'equity_any_neg_in_window', 'ROE_12m', 'LDR', 'loans_avg_Q', 'loans_any_neg_in_Q', 'LoanYield_3m', 'flag_roe_outlier', 'flag_yield_outlier', 'flag_ldr_outlier']
regn
bank_1     73
bank_10    73
bank_11    73
dtype: int64


In [22]:
print("index name:", clean.index.name, "index names:", clean.index.names)
print("index sample:", clean.index[:3].tolist())
print("df regn sample:", df['regn'].head(3).tolist())
# Принудительное восстановление:
clean = clean.reset_index(drop=True)
df_sorted = df.sort_values(['regn','date']).reset_index(drop=True)
clean['regn'] = df_sorted['regn'].values
print("after:", 'regn' in clean.columns, clean['regn'].nunique())


index name: None index names: [None]
index sample: [0, 1, 2]
df regn sample: ['bank_1', 'bank_1', 'bank_1']
after: True 20


In [23]:
# YTD-версия: восстановление потоков из накопленного итога с reset в январе
# Предположение: NIC(1 янв Y) = FY(Y-1); YTD_Y(1 янв Y) = 0 (старт нового года)

def _monthly_profit_ytd(g):
    g = g.sort_values('date').copy()
    nic = g['Net_Income_Current'].values
    dates = pd.to_datetime(g['date']).values
    flows = np.full(len(g), np.nan)
    for i in range(1, len(g)):
        d, prev = pd.Timestamp(dates[i]), pd.Timestamp(dates[i-1])
        if prev != d - pd.DateOffset(months=1):
            continue
        # flow месяца, завершившегося перед датой d
        if d.month == 2:
            # январь Y: YTD_Y(1 фев) - 0
            flows[i] = nic[i]
        else:
            # февраль..декабрь: YTD_Y(t) - YTD_Y(t-1m); для d=1 янв Y+1: FY(Y) - YTD_Y(1 дек)
            flows[i] = nic[i] - nic[i-1]
    g['profit_month_ytd'] = flows
    return g

def _quarterly_nii_ytd(g):
    g = g.sort_values('date').copy()
    nii = g['Net_Interest_Income'].values
    dates = pd.to_datetime(g['date']).values
    qnii = np.full(len(g), np.nan)
    for i in range(len(g)):
        d = pd.Timestamp(dates[i])
        if d.month == 4:                         # Q1
            qnii[i] = nii[i]
        elif d.month in (7, 10):                 # Q2, Q3
            # нужна точка 3 мес назад
            if i >= 3 and pd.Timestamp(dates[i-3]) == d - pd.DateOffset(months=3):
                qnii[i] = nii[i] - nii[i-3]
        elif d.month == 1 and i >= 3:            # Q4 (точка 1 янв Y+1)
            if pd.Timestamp(dates[i-3]) == d - pd.DateOffset(months=3):
                qnii[i] = nii[i] - nii[i-3]
    g['qnii_ytd'] = qnii
    return g

clean = pd.concat([_monthly_profit_ytd(g) for _, g in clean.groupby('regn')]).reset_index(drop=True)
clean = pd.concat([_quarterly_nii_ytd(g) for _, g in clean.groupby('regn')]).reset_index(drop=True)


g = clean.groupby('regn', group_keys=False)

# ROE 12m (YTD): rolling-12 сумма месячных потоков / rolling-13 среднее Equity
clean['profit_12m_ytd'] = g['profit_month_ytd'].apply(lambda s: s.rolling(12, min_periods=12).sum())
# equity_avg_13 уже посчитан во flow-версии, переиспользуем
clean['ROE_12m_ytd'] = np.where(
    clean['equity_any_neg_in_window'] | clean['equity_avg_13'].isna() | clean['profit_12m_ytd'].isna(),
    np.nan,
    clean['profit_12m_ytd'] / clean['equity_avg_13']
)

# LoanYield 3m (YTD): 4 * qnii / среднее Loans_Total_Net по 4 точкам квартала
clean['loans_avg_4'] = g['Loans_Total_Net'].apply(lambda s: s.rolling(4, min_periods=4).mean())
clean['loans_any_neg_in_q'] = g['Loans_Total_Net'].apply(lambda s: (s < 0).rolling(4, min_periods=4).sum() > 0)
clean['LoanYield_3m_ytd'] = np.where(
    clean['qnii_ytd'].isna() | clean['loans_any_neg_in_q'] | clean['loans_avg_4'].isna(),
    np.nan,
    4 * clean['qnii_ytd'] / clean['loans_avg_4']
)

# Сравнение flow vs YTD — сводная статистика
print("ROE_12m  (flow):", clean['ROE_12m'].describe()[['count','50%','mean','max']].to_dict())
print("ROE_12m  (ytd) :", clean['ROE_12m_ytd'].describe()[['count','50%','mean','max']].to_dict())
print("LoanYield (flow):", clean['LoanYield_3m'].describe()[['count','50%','mean','max']].to_dict())
print("LoanYield (ytd) :", clean['LoanYield_3m_ytd'].describe()[['count','50%','mean','max']].to_dict())


ROE_12m  (flow): {'count': 1032.0, '50%': 0.874622066425199, 'mean': 1.107931517994442, 'max': 12.167754064045726}
ROE_12m  (ytd) : {'count': 1031.0, '50%': 0.14971357258616655, 'mean': 0.22613844456511767, 'max': 3.8575939359673415}
LoanYield (flow): {'count': 464.0, '50%': 0.07099281958516276, 'mean': 0.16527952928769712, 'max': 5.664553709097139}
LoanYield (ytd) : {'count': 462.0, '50%': 0.008867101069431402, 'mean': 0.06799176254093918, 'max': 4.484389099440534}


In [24]:
print("clean shape:", clean.shape, "columns:", clean.columns.tolist())
print("df shape:", df.shape)



clean shape: (1460, 35) columns: ['date', 'regn', 'Assets', 'Loans_Total_Net', 'Loans_LLP', 'Equity', 'Net_Income_Current', 'Client_Deposits', 'Net_Interest_Income', 'month', 'flag_equity_negative', 'flag_loans_negative', 'flag_deposits_negative', 'flag_assets_negative', 'flag_llp_exceeds_net', 'flag_llp_positive_sign', 'Loans_Gross', 'profit_12m', 'equity_avg_13', 'equity_any_neg_in_window', 'ROE_12m', 'LDR', 'loans_avg_Q', 'loans_any_neg_in_Q', 'LoanYield_3m', 'flag_roe_outlier', 'flag_yield_outlier', 'flag_ldr_outlier', 'profit_month_ytd', 'qnii_ytd', 'profit_12m_ytd', 'ROE_12m_ytd', 'loans_avg_4', 'loans_any_neg_in_q', 'LoanYield_3m_ytd']
df shape: (1460, 10)


In [25]:
b = clean[(clean['regn']=='bank_1') & clean['Net_Interest_Income'].notna()].sort_values('date')
print(b[['date','Net_Interest_Income']].tail(12))


         date  Net_Interest_Income
36 2023-01-01              15793.8
39 2023-04-01              14339.0
42 2023-07-01              15079.4
45 2023-10-01              17322.1
48 2024-01-01              17514.8
51 2024-04-01              17625.6
54 2024-07-01              16920.9
57 2024-10-01              18119.9
60 2025-01-01              20197.3
63 2025-04-01              17481.0
66 2025-07-01              16306.0
72 2026-01-01              13631.7
